# 06_xgboost.ipynb

Converted from your cell-by-cell rebuild guide.

## Cell 1 — Imports

In [2]:
%pip install shap

  Using cached shap-0.51.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (25 kB)
Using cached shap-0.51.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (1.1 MB)
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install --force-reinstall --no-cache-dir "numpy==1.26.4" "pandas==1.5.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 396.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 480.6 MB/s  0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2023.3.post1
    Uninstalling pytz-2023.3.post1:
      Successfully uninstalled pytz-2023.3.post1
  Attempting uninstall: six━━━━━━━━━━━━━━━━━━━━━ 0/5 [pytz]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/5 [pytz]    WARNING: Ignoring invalid distribution ~cipy (/opt/conda/lib/python3.11/site-packages)
    Found existing installation: six 1.16.0
    Uninstalling six-1.16.0:━━━━━━━━━━━━━━━━ 0/5 [pytz]
      Successfully uninstalled six-1.16.0━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [six]
  Attempting uninstall: numpy0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [six]
   ━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [six]    WARNING: Ignoring invalid distribution ~cipy (/opt/conda/lib/python3.11/site-packages)
    Found existing installation: numpy 2.4.3
    Uninstalling numpy-2.4.3:╺━━━━━━━

In [1]:
import numpy as np
import pandas as pd

print("numpy:", np.__version__)
print("pandas:", pd.__version__)

numpy: 1.26.4
pandas: 1.5.3


In [3]:
import pandas as pd
import numpy as np
import joblib, json
from pathlib import Path

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

try:
    import shap
    SHAP_AVAILABLE = True
    print("SHAP available")
except Exception as e:
    SHAP_AVAILABLE = False
    print("SHAP not available:", e)

COINS = ["Bitcoin", "Ethereum", "Dogecoin"]

SHAP available


## Cell 2 — Train XGBoost per coin with class imbalance handling

In [4]:
for coin in COINS:
    feat_cols = load_feature_cols(coin)
    df = pd.read_csv(f"features_{coin.lower()}.csv", parse_dates=["date"]).sort_values("date")
    train, val, test = time_split(df)

    X_train = train[feat_cols].values;  y_train = train["target"].values.astype(int)
    X_val   = val[feat_cols].values;    y_val   = val["target"].values.astype(int)
    X_test  = test[feat_cols].values;   y_test  = test["target"].values.astype(int)

    # XGBoost handles imbalance via scale_pos_weight instead of SMOTE
    neg_count = (y_train == 0).sum()
    pos_count = (y_train == 1).sum()
    scale_pos_weight = neg_count / pos_count
    print(f"{coin}: scale_pos_weight = {scale_pos_weight:.2f} | features={len(feat_cols)}")

    param_dist = {
        "n_estimators": [100, 200, 400, 600],
        "max_depth": [3, 4, 5, 6, 7],
        "learning_rate": [0.01, 0.05, 0.1, 0.2],
        "subsample": [0.6, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "reg_alpha": [0, 0.1, 0.5, 1.0],
        "reg_lambda": [1.0, 2.0, 5.0],
        "min_child_weight": [1, 3, 5],
    }
    tscv = TimeSeriesSplit(n_splits=4)
    xgb_base = xgb.XGBClassifier(
        objective="binary:logistic",
        scale_pos_weight=scale_pos_weight,
        eval_metric="aucpr",
        use_label_encoder=False,
        random_state=42,
        tree_method="hist",
    )
    rs = RandomizedSearchCV(
        xgb_base, param_dist, n_iter=30, cv=tscv,
        scoring="f1", random_state=42, refit=True, n_jobs=-1, verbose=0
    )
    rs.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    best_xgb = rs.best_estimator_
    print(f"{coin} best params: {rs.best_params_}")

    # Threshold tuning on val
    val_prob = best_xgb.predict_proba(X_val)[:, 1]
    best_thr = max(
        np.linspace(0.3, 0.7, 41),
        key=lambda t: f1_score(y_val, (val_prob >= t).astype(int))
    )

    # Evaluate on test
    test_prob = best_xgb.predict_proba(X_test)[:, 1]
    y_pred = (test_prob >= best_thr).astype(int)
    prec, rec, _ = precision_recall_curve(y_test, test_prob)
    pr_auc = auc(rec, prec)
    roc = roc_auc_score(y_test, test_prob)
    f1 = f1_score(y_test, y_pred)

    print(f"\n{coin} XGBoost TEST:")
    print(classification_report(y_test, y_pred, target_names=["Down", "Up"]))
    print(f"ROC-AUC={roc:.4f}  PR-AUC={pr_auc:.4f}  F1(Up)={f1:.4f}")

    xgb_models[coin] = {
        "model": best_xgb,
        "threshold": best_thr,
        "feature_cols": feat_cols,
        "metrics": {"f1_up": f1, "roc_auc": roc, "pr_auc": pr_auc},
    }

print("\n✅ XGBoost complete for all coins.")


Bitcoin: scale_pos_weight = 1.11 | features=60
Bitcoin best params: {'subsample': 0.6, 'reg_lambda': 5.0, 'reg_alpha': 0.5, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 0.6}

Bitcoin XGBoost TEST:
              precision    recall  f1-score   support

        Down       0.45      0.27      0.34       277
          Up       0.54      0.72      0.62       329

    accuracy                           0.51       606
   macro avg       0.49      0.49      0.48       606
weighted avg       0.50      0.51      0.49       606

ROC-AUC=0.4838  PR-AUC=0.5516  F1(Up)=0.6154
Ethereum: scale_pos_weight = 1.05 | features=51
Ethereum best params: {'subsample': 0.8, 'reg_lambda': 5.0, 'reg_alpha': 1.0, 'n_estimators': 400, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.6}

Ethereum XGBoost TEST:
              precision    recall  f1-score   support

        Down       0.43      0.29      0.35       123
        

## Cell 3 — SHAP explainability (XGBoost)

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, coin in enumerate(COINS):
    feat_cols = xgb_models[coin]["feature_cols"]
    df = pd.read_csv(f"features_{coin.lower()}.csv", parse_dates=["date"]).sort_values("date")
    _, _, test = time_split(df)
    X_test = test[feat_cols].values

    explainer = shap.TreeExplainer(xgb_models[coin]["model"])
    shap_vals = explainer.shap_values(X_test)

    # Summary plot — saved separately per coin
    plt.figure(figsize=(8, 6))
    shap.summary_plot(
        shap_vals, X_test, feature_names=feat_cols,
        show=False, max_display=15
    )
    plt.title(f"{coin} — XGBoost SHAP Summary")
    plt.tight_layout()
    plt.savefig(f"xgb_shap_{coin.lower()}.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved xgb_shap_{coin.lower()}.png")


Saved xgb_shap_bitcoin.png
Saved xgb_shap_ethereum.png
Saved xgb_shap_dogecoin.png


## Cell 4 — Save XGBoost models

In [6]:
for coin in COINS:
    joblib.dump(xgb_models[coin]["model"], f"xgb_model_{coin.lower()}.joblib")
    print(f"Saved xgb_model_{coin.lower()}.joblib  threshold={xgb_models[coin]['threshold']:.2f}")

Saved xgb_model_bitcoin.joblib  threshold=0.30
Saved xgb_model_ethereum.joblib  threshold=0.30
Saved xgb_model_dogecoin.joblib  threshold=0.30
